# Модуль 8. Разбор кейсов: Скрининг с журналом и контролями

**Модуль 8. Автор/аффилиация:** автор курса уточняет перед публикацией. **Среда:** WSL/uv/Python 3.12. **Данные:** molecules.csv плюс заданные учебные контроли μ=100/10, σ=5/5 и библиотека N=1000,A=20,k=50,a=5.

Развести качество ассая, структурный риск и ранжирование; не удалять спорные записи молча.

**Workflow:** Структуры → предупреждения → статусы → Z′ → EF → следующий контроль.

**Результаты обучения:** объяснить вход/политику; воспроизвести расчёт и отрицательный контроль; ограничить научный вывод.

Оцените результат до запуска. После разбора переходите во [второй notebook](lab08_practice.ipynb). [Установка среды](../SETUP.md).

[Связанная глава](../../Module_theory/Module%208_Screening.md).

## 0. Подготовка и паспорт запуска

Запускайте ячейки сверху вниз после Restart Kernel. Путь Python должен указывать на проектную `.venv`; устанавливать пакеты в ячейке не нужно. Эта ячейка только определяет пути и создаёт отдельный каталог результатов разбора.

In [1]:
from pathlib import Path
import sys, json, math, csv, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'Labs/chemistry.py').is_file())
for folder in [ROOT/'Stepik_course/practice', ROOT/'Labs']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
DATA = ROOT/'Labs/datasets'
OUT = ROOT/'Labs/results/lab08/worked'
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
print('Python:', sys.executable)


Python: /home/miakhlov/course-notebooks/.venv/bin/python


## 1. Предупреждение и решение

Скрининг «Химеры» должен сохранять все исходные ID. PAINS — структурный сигнал для проверки возможной помехи, не доказанный механизм и не автоматическое основание удалить молекулу. Различаем invalid, review и eligible: последний статус означает только отсутствие заданных предупреждений.

**Перед запуском:** Получите все PAINS-предупреждения, не объявляя механизм доказанным.

In [2]:
from chemistry import screen
from metrics import z_prime, enrichment
raw=pd.read_csv(DATA/'molecules.csv',keep_default_na=False)
records=[{'id':r.id,**screen(r.smiles)} for r in raw.itertuples()]
table=pd.DataFrame(records)
display(table[['id','status','reason']])
assert len(table)==12


[14:00:08] Initializing MetalDisconnector
[14:00:08] Running MetalDisconnector
[14:00:08] Initializing Normalizer
[14:00:08] Running Normalizer
[14:00:08] Initializing MetalDisconnector
[14:00:08] Running MetalDisconnector
[14:00:08] Initializing Normalizer
[14:00:08] Running Normalizer
[14:00:08] Running LargestFragmentChooser
[14:00:08] Running Uncharger
[14:00:08] Initializing MetalDisconnector
[14:00:08] Running MetalDisconnector
[14:00:08] Initializing Normalizer
[14:00:08] Running Normalizer
[14:00:08] Initializing MetalDisconnector
[14:00:08] Running MetalDisconnector
[14:00:08] Initializing Normalizer
[14:00:08] Running Normalizer
[14:00:08] Running LargestFragmentChooser
[14:00:08] Running Uncharger
[14:00:08] Initializing MetalDisconnector
[14:00:08] Running MetalDisconnector
[14:00:08] Initializing Normalizer
[14:00:08] Running Normalizer
[14:00:08] Initializing MetalDisconnector
[14:00:08] Running MetalDisconnector
[14:00:08] Initializing Normalizer
[14:00:08] Running Norma

,id,status,reason
0,M01,eligible,No listed alert; activity remains unknown
1,M02,eligible,No listed alert; activity remains unknown
2,M03,eligible,No listed alert; activity remains unknown
3,M04,eligible,No listed alert; activity remains unknown
4,M05,eligible,No listed alert; activity remains unknown
5,M06,review,PAINS:catechol_A(92)
6,M07,eligible,No listed alert; activity remains unknown
7,M08,invalid,Invalid SMILES
8,M09,invalid,SMILES must be a nonempty string
9,M10,eligible,No listed alert; activity remains unknown


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Предупреждение и решение — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 2. Три разных уровня контроля

Проверка катехола и пустого ввода относится к обработке структуры. Z′=1−3(σp+σn)/|μp−μn| описывает разделение контрольных распределений. EF=(a/k)/(A/N) относится к обогащению ранжирования при известной разметке. Хороший Z′ не подтверждает эффективность конкретного лиганда.

**Перед запуском:** Посчитайте Z′ и EF на отдельных числах.

In [3]:
good=z_prime(100,10,5,5)
bad=z_prime(100,10,20,20)
ef=enrichment(1000,20,50,5)
print({'Z_good':good,'Z_bad':bad,'EF_50':ef})
assert math.isclose(good,2/3) and math.isclose(bad,-1/3)
assert ef==5


{'Z_good': 0.6666666666666667, 'Z_bad': -0.33333333333333326, 'EF_50': 5.0}


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Три разных уровня контроля — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 3. Журнал вместо молчаливого фильтра

Сохраняем status и reason для каждой записи, затем связываем review со следующим контролем. Число EF бессмысленно без истинного числа активных в библиотеке. Отрицательный Z′ сигнализирует слабое разделение контролей в выбранной оценке, но не отрицательную активность соединения.

**Перед запуском:** Запишите следующий контроль для review и объяснение для eligible.

In [4]:
assert screen('Oc1ccccc1O')['status']=='review'
assert screen('')['status']=='invalid'
table.to_csv(OUT/'decisions.csv',index=False)
artifact={'Z_prime':good,'EF_at_50':ef,'statuses':table.status.value_counts().to_dict()}


[14:00:08] Initializing MetalDisconnector
[14:00:08] Running MetalDisconnector
[14:00:08] Initializing Normalizer
[14:00:08] Running Normalizer
[14:00:08] Initializing MetalDisconnector
[14:00:08] Running MetalDisconnector
[14:00:08] Initializing Normalizer
[14:00:08] Running Normalizer
[14:00:08] Running LargestFragmentChooser
[14:00:08] Running Uncharger


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Журнал вместо молчаливого фильтра — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 4. Артефакт и граница доказательства

Катехол review, пустой ввод invalid; Z′=2/3; EF@50=5; журнал 12 строк.

Следующая ячейка сохраняет именно новый запуск и хеши входов. Отсутствующее вычисление нельзя заменять старым output. Время относится к этому компьютеру и не является обещанием производительности.

In [5]:
record={'module':8,'scope':'worked reference execution','python':sys.executable,'elapsed_seconds':time.perf_counter()-started,'result':artifact}
(OUT/'benchmark.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(OUT/'benchmark.json')

/home/miakhlov/course-notebooks/Labs/results/lab08/worked/benchmark.json


## Источники и следующий шаг

1. Исходный вычислительный пример: [Labs/chemistry.py](../chemistry.py), [описание данных](../datasets/README.md).
2. [RDKit: Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html).
3. [Карта чтения](../../Literature/READING_MAP.md).

Перейдите к [lab08_practice.ipynb](lab08_practice.ipynb). Через неделю повторите первый кейс без просмотра ключа.